# 📅 2026-05-19 (화) 개발 노트 : Hidden Gem 헤더 이미지 파이프라인 + 프론트엔드 MVP 착수

## 🎯 오늘의 목표
- [x] Steam 헤더 이미지 4,190개 일괄 채우기 (Phase A/B/C)
- [x] pgvector 임베딩 적재 + HNSW 인덱스 생성
- [x] 추천엔진 하이브리드 v3 (49차원 지표 + 1536차원 임베딩)
- [x] 비게임 데이터(유틸리티 앱 등) DB 비활성화
- [x] Next.js 15 프론트엔드 세팅 + 24개 파일 작성
- [x] 프론트-백엔드 API 연결 및 런타임 점검

## 🛠 진행 상황 및 핵심 기록

### 헤더 이미지 파이프라인
- **Phase A** `scripts/fill_header_images.py`: app_id 기반 Steam CDN URL 일괄 생성 → 4,190개 완료
  - CDN URL 패턴: `https://cdn.cloudflare.steamstatic.com/steam/apps/{app_id}/header.jpg`
- **Phase B** `scripts/verify_header_images.py`: HEAD 요청으로 404 탐지 → 391개 실패 발견
  - 실패 원인: 최근 게임(app_id 3,000,000+)은 Akamai CDN 이용, 해시값 포함 URL 패턴 상이
- **Phase C** `scripts/fix_failed_headers.py`: Steam API로 실제 URL 복구
  - `store.steampowered.com/api/appdetails?appids={app_id}` → `header_image` 필드 추출
  - 한국 IP + rate limit 이슈로 390개 복구 불가 → `onError` fallback으로 프론트 처리
  - `concurrency=1`로 낮췄을 때 1개 복구 성공 확인

### pgvector 임베딩
- `game_metrics` 테이블에 `embedding vector(1536)` 컬럼 추가
- `scripts/load_embeddings.py`: pkl 파일(JSON string)에서 1536차원 파싱 → raw SQL UPDATE
  - SQLAlchemy ORM `update(GameMetric).values(embedding=...)` 은 컬럼 미정의로 에러
  - `text("UPDATE game_metrics SET embedding = :emb WHERE game_id = :gid")` 로 우회
- HNSW 인덱스: `vector_cosine_ops`, m=16, ef_construction=64
- 적재 완료: **4,141개** (비활성 49개 제외)

### 추천엔진 하이브리드 v3
- `fastapi_app/services/recommender.py` 업데이트
- 기존: 코사인(49차원) 50% + 유클리드(49차원) 50%
- 변경: **코사인 35% + 유클리드 35% + 임베딩 코사인 30%**
- 임베딩 없는 게임은 자동으로 5:5 fallback
- 검증: Undertale → DELTARUNE 97.09% 매칭 확인

### 비게임 데이터 정리
- `genres ILIKE '%유틸리티%'` 등 필터로 26개 비활성화
- `is_active = false` 처리 (물리 삭제 X)

### 프론트엔드 MVP
- **스택**: Next.js 16.2.6 (App Router, Turbopack) + TypeScript + Tailwind CSS v4
- **패키지**: @tanstack/react-query, framer-motion, zustand, axios, @radix-ui/*, lucide-react
- **파일 구조**: 24개 파일 작성 (api.ts, hooks, store, ui 컴포넌트, layout, pages)
- **디자인**: 퍼플 액센트 #7C3AED, Linear.app 감성, 라이트/다크 모드

### 핵심 API 엔드포인트
```
GET  /api/v1/games/search?q={query}&limit={n}
GET  /api/v1/games/{app_id}
GET  /api/v1/games/stats/overview
POST /api/v1/games/recommend/by-game      body: {app_id, count}
POST /api/v1/games/recommend/by-preference body: {preferences: Record<string,number>, count}
```

### preferences 필드명 주의사항
- FastAPI에서 `NUMERIC_METRIC_FIELDS` 기준으로 유효성 검사
- 반드시 실제 49개 지표명 사용 (예: `narrative_depth`, `cozy_factor`)
- ❌ `story_richness`, `artistic_value` 등 임의 필드명 → 400 Bad Request

## 🚨 트러블슈팅

### 1. SearchBar placeholder 순환 미동작
- **문제**: placeholder가 2.8초마다 바뀌지 않음
- **원인**: React Strict Mode의 이중 마운트로 interval이 꼬임
- **해결**: `setPhIdx((i) => ...)` 대신 클로저 변수 `let idx` 사용
```typescript
useEffect(() => {
  let idx = 0;
  const t = setInterval(() => {
    idx = (idx + 1) % SEARCH_PLACEHOLDERS.length;
    setPhIdx(idx);
  }, 2800);
  return () => clearInterval(t);
}, []);
```

### 2. SQLAlchemy ORM embedding 컬럼 에러
- **문제**: `update(GameMetric).values(embedding=str(emb))` → `CompileError: Unconsumed column names`
- **원인**: ORM 모델에 `embedding` 컬럼 미정의
- **해결**: `pgvector.sqlalchemy.Vector` import 후 모델에 추가, 로드 스크립트는 raw SQL로 우회

### 3. 프론트 400 Bad Request
- **문제**: `recommend/by-preference` 호출 시 400 에러
- **원인**: `DEFAULT_PREFS`에 존재하지 않는 지표명 사용 (`story_richness` 등)
- **해결**: 실제 49개 지표명으로 교체
```typescript
const DEFAULT_PREFS = {
  narrative_depth: 8,
  lore_richness: 7,
  art_style_uniqueness: 7,
  replay_value: 6,
  exploration_reward: 7,
  soundtrack_impact: 7,
}
```

### 4. Docker DB 연결 실패
- **문제**: FastAPI 시작 후 DB 연결 `OSError: Connect call failed`
- **원인**: Docker 컨테이너 `hidden_gem_db` 미실행
- **해결**: `docker start hidden_gem_db` 후 FastAPI 재시작

### 5. 가상환경 미활성화
- **문제**: `ModuleNotFoundError: No module named 'fastapi'`
- **해결**: `source .venv/Scripts/activate` 후 실행

## 💡 인사이트 및 다음 할 일

### 배운 점
- Steam CDN URL 패턴이 구버전(cloudflare)과 신버전(akamai+hash)으로 이원화됨
- pgvector HNSW 인덱스는 `vector_cosine_ops` 지정 필수
- React Strict Mode에서 `useEffect` 이중 실행 → 클로저 변수로 interval 안전하게 관리
- Next.js 캐시(`.next`) 문제 시 `rm -rf .next` 후 재시작
- FastAPI 지표명 유효성 검사는 `NUMERIC_METRIC_FIELDS` 기준 → 프론트와 필드명 반드시 일치

### 다음 할 일
- [ ] 게임 상세 페이지 (`/game/[appId]`) 런타임 점검
- [ ] 랭킹 페이지 전체/장르별 탭 동작 확인
- [ ] 취향 분석 슬라이더 페이지 동작 확인
- [ ] 다크모드 토글 동작 확인
- [ ] 모바일 반응형 점검
- [ ] Steam 로그인 구현 (django-social-auth)
- [ ] '취향 DNA' 공유 카드 구현 (바이럴 핵심)
- [ ] Vercel + Railway 배포